# Kafka Demo

## Important: Connect to Kafka Broker Server FIRST

**Before running any code in this notebook**, establish an SSH tunnel to the Kafka server:

```
ssh -L <local_port>:localhost:<remote_port> <user>@<remote_server> -NTf
```

Find connection details (remote_server, user, password, ports) on the Canvas lab 2 assignment page.

**Verify your connection is active:**
```bash
lsof -i :<local_port>  # Should show an ssh process
```

**To kill the connection when done:**
```
lsof -ti:<local_port> | xargs kill -9
```

---

## Setup

It is recommended to set up a python environment for this lab (and all other ones).
```
python -m venv <environment_name>
source <environment_name>/bin/activate  # On Windows: <environment_name>\Scripts\activate
```

Then install the requirements:
```
pip install -r requirements.txt
```
Or manually:
```
pip install kafka-python
```

Al usar docker compose, el tunel ya se establece al levantar el docker compose, por tanto no se necesita crear por fuera

In [1]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer
from typing import Dict, Any

# Reemplaza con tu andrew_id o cualquier identificador único
andrew_id = "pporras"  # <-- cambia esto por tu identificador
topic = f"lab02-{andrew_id}"
print(f"Topic: {topic}")

Topic: lab02-pporras


### Producer Mode -> Writes Data to Broker

In [ ]:
# Below schema is for messages. You may change the city data if you wish but it is optional.
def make_city_data(city: str, temperature_f: str) -> Dict[str, Any]:
    return {
        "city": city,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature_f": temperature_f, # temperature in fahrenheit
    }

In [ ]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

producer = KafkaProducer(
    bootstrap_servers=["localhost:9092"],
    value_serializer=lambda x: dumps(x).encode('utf-8')
)

In [ ]:
cities = [
    make_city_data("Pittsburgh", 64),
    make_city_data("New York", 58),
    make_city_data("Los Angeles", 75),
]

print("Writing to Kafka Broker")
for i in range(10):
    data = cities[randint(0, len(cities)-1)]  # random selection
    producer.send(topic=topic, value=data)
    sleep(1)

producer.flush()
print(f"Data written to topic: {topic}")

### Consumer Mode -> Reads Data from Broker

In [ ]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

consumer = KafkaConsumer(
    topic,
    bootstrap_servers=["localhost:9092"],
    auto_offset_reset='earliest',  # lee desde el primer mensaje del topic
    enable_auto_commit=True,
    auto_commit_interval_ms=1000
)

print('Reading Kafka Broker')
for message in consumer:
    message_str = message.value.decode('utf-8')
    message_dict = loads(message_str)
    print(message_dict)
    os.system(f"echo {message_str} >> kafka_log.csv")

# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
# Comando kcat para consumir los primeros 5 mensajes del topic desde Docker
# Ejecuta esto en tu terminal (no en el notebook):
#
# docker exec kcat kcat -b kafka:9092 -t lab02-your_id -C -o earliest -c 5 -f "%o: %s\n"
#
# Explicación de flags:
#   -b kafka:9092   -> broker (nombre del servicio en Docker Compose)
#   -t lab02-your_id -> topic (reemplaza your_id por tu identificador)
#   -C              -> modo consumer
#   -o earliest     -> empieza desde el offset 0 (primer mensaje)
#   -c 5            -> consume exactamente 5 mensajes y para
#   -f "%o: %s\n"   -> formato: "offset: mensaje"
#
# El OFFSET es el número secuencial de cada mensaje dentro del topic.
# Permite que un consumer retome desde donde lo dejó si se desconecta.